
# 02b – Weather Data Cleaning

## Objective
The weather data for Zurich from 2023 to 2025
will be loaded, examined, cleaned, and prepared
for integration with the bicycle traffic data.

## Procedure
1. Load the weather data
2. Examine data quality
3. Handle missing values
4. Check duplicate records
5. Standardize timestamps and data types
6. Create additional weather variables
7. Save the cleaned dataset


In [2]:

import pandas as pd
from pathlib import Path

# Find the project root directory
current = Path.cwd().resolve()

project_root = next(
    (
        p for p in [current, *current.parents]
        if (p / "requirements.txt").exists()
        and (p / "notebooks").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Project root directory not found!"
    )

# Define the path to the raw weather data
file_path = (
    project_root
    / "data"
    / "raw"
    / "weather_zurich_hourly_2023_2025.csv"
)

# Load the CSV file
df_weather = pd.read_csv(file_path)

# Display the first five rows
df_weather.head()


,time,temperature_2m,precipitation,relative_humidity_2m,wind_speed_10m,cloud_cover
0,2023-01-01T00:00,8.1,0.0,84,5.1,74
1,2023-01-01T01:00,8.2,0.0,83,4.6,48
2,2023-01-01T02:00,8.4,0.0,85,3.6,48
3,2023-01-01T03:00,8.0,0.0,86,2.9,61
4,2023-01-01T04:00,8.0,0.0,86,3.2,41



## 1. Initial Data Quality Assessment

Before cleaning the dataset, we examine its
structure and quality.

The analysis includes:
- Dataset dimensions
- Column names and data types
- Missing values
- Duplicate records
- Descriptive statistics

This assessment helps identify potential
data quality issues that need to be addressed
during the cleaning process.


In [3]:

# Display dataset dimensions
print("Dataset dimensions:")
print(df_weather.shape)

# Display column names
print("\nColumn names:")
print(df_weather.columns.tolist())

# Display data types
print("\nData types:")
print(df_weather.dtypes)

# Check for missing values
print("\nMissing values:")
print(df_weather.isnull().sum())

# Check for duplicate records
print("\nDuplicate records:")
print(df_weather.duplicated().sum())

# Display descriptive statistics
print("\nDescriptive statistics:")
display(df_weather.describe())


Dataset dimensions:
(26304, 6)

Column names:
['time', 'temperature_2m', 'precipitation', 'relative_humidity_2m', 'wind_speed_10m', 'cloud_cover']

Data types:
time                        str
temperature_2m          float64
precipitation           float64
relative_humidity_2m      int64
wind_speed_10m          float64
cloud_cover               int64
dtype: object

Missing values:
time                    0
temperature_2m          0
precipitation           0
relative_humidity_2m    0
wind_speed_10m          0
cloud_cover             0
dtype: int64

Duplicate records:
0

Descriptive statistics:


,temperature_2m,precipitation,relative_humidity_2m,wind_speed_10m,cloud_cover
count,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000
mean,11.725152,0.137203,78.544442,6.603836,64.846107
std,7.902187,0.378188,15.580745,4.567379,34.906086
min,-7.700000,0.000000,23.000000,0.000000,0.000000
25%,5.700000,0.000000,70.000000,3.300000,35.000000
50%,11.200000,0.000000,83.000000,5.600000,77.000000
75%,17.400000,0.100000,91.000000,8.800000,98.000000
max,35.900000,6.600000,100.000000,34.300000,100.000000



## 2. Handling Missing Values

The initial data quality assessment shows that
the dataset contains no missing values.

All 26,304 hourly observations contain complete
weather information for the selected variables.

Therefore, no imputation or removal of missing
values is necessary.

The original weather measurements are preserved.


In [4]:

# Calculate the total number of missing values
total_missing = df_weather.isnull().sum().sum()

print("Total missing values:", total_missing)

# Check whether missing value treatment is required
if total_missing == 0:
    print("No missing values found.")
    print("No imputation or row removal required.")
else:
    print("Missing values require further investigation.")


Total missing values: 0
No missing values found.
No imputation or row removal required.



## 3. Standardizing Timestamps

The time column is currently stored as a string.

To enable time-based analysis and integration
with the bicycle traffic dataset, timestamps
are converted to datetime objects.

All timestamps are interpreted as UTC because
the weather data was requested using the UTC
timezone.

The following checks are performed:
- Convert timestamps to datetime
- Check for invalid timestamps
- Check for duplicate timestamps
- Verify the hourly time series
- Identify missing hourly timestamps


In [5]:

# Convert the time column to UTC datetime
df_weather["time"] = pd.to_datetime(
    df_weather["time"],
    utc=True,
    errors="coerce"
)

# Check for invalid timestamps
invalid_timestamps = df_weather["time"].isna().sum()

# Check for duplicate timestamps
duplicate_timestamps = df_weather["time"].duplicated().sum()

# Sort the dataset chronologically
df_weather = df_weather.sort_values(
    "time"
).reset_index(drop=True)

# Create the expected hourly time range
expected_hours = pd.date_range(
    start="2023-01-01 00:00:00",
    end="2025-12-31 23:00:00",
    freq="h",
    tz="UTC"
)

# Identify missing hourly timestamps
missing_hours = expected_hours.difference(
    df_weather["time"].dropna()
)

# Display validation results
print("Time column data type:")
print(df_weather["time"].dtype)

print("\nInvalid timestamps:", invalid_timestamps)
print("Duplicate timestamps:", duplicate_timestamps)
print("Missing hourly timestamps:", len(missing_hours))

print("\nFirst timestamp:")
print(df_weather["time"].min())

print("\nLast timestamp:")
print(df_weather["time"].max())


Time column data type:
datetime64[us, UTC]

Invalid timestamps: 0
Duplicate timestamps: 0
Missing hourly timestamps: 0

First timestamp:
2023-01-01 00:00:00+00:00

Last timestamp:
2025-12-31 23:00:00+00:00



## 4. Creating Additional Weather Variables

Additional weather variables are created to
support the analysis of bicycle traffic patterns.

The following variables are added:

1. **is_rainy**: Indicates whether precipitation
   occurred during a specific hour.

2. **temperature_category**: Groups temperatures
   into predefined ranges.

These variables will help compare bicycle traffic
under different weather conditions.

The original weather measurements are preserved.


In [6]:

# Create a binary variable for precipitation
# True = precipitation occurred
# False = no precipitation

df_weather["is_rainy"] = (
    df_weather["precipitation"] > 0
)

# Count rainy and dry hours
print("Rainy and dry hours:")
print(df_weather["is_rainy"].value_counts())

# Calculate percentages
print("\nPercentage distribution:")
print(
    df_weather["is_rainy"]
    .value_counts(normalize=True) * 100
)


Rainy and dry hours:
is_rainy
False    18904
True      7400
Name: count, dtype: int64

Percentage distribution:
is_rainy
False    71.867397
True     28.132603
Name: proportion, dtype: float64


In [7]:

# Define temperature category boundaries
temperature_bins = [
    float("-inf"),
    0,
    10,
    20,
    float("inf")
]

# Define category labels
temperature_labels = [
    "Below 0°C",
    "0-10°C",
    "10-20°C",
    "20°C and above"
]

# Create temperature categories
df_weather["temperature_category"] = pd.cut(
    df_weather["temperature_2m"],
    bins=temperature_bins,
    labels=temperature_labels,
    right=False
)

# Display temperature category distribution
print("Temperature categories:")
print(
    df_weather["temperature_category"]
    .value_counts()
    .sort_index()
)


Temperature categories:
temperature_category
Below 0°C          1379
0-10°C            10220
10-20°C           10431
20°C and above     4274
Name: count, dtype: int64


In [8]:

# Display the new weather variables
display(
    df_weather[
        [
            "time",
            "temperature_2m",
            "precipitation",
            "is_rainy",
            "temperature_category"
        ]
    ].head(10)
)

# Check for missing values in the new columns
print("\nMissing values in new variables:")
print(
    df_weather[
        ["is_rainy", "temperature_category"]
    ].isnull().sum()
)

# Display updated dataset dimensions
print("\nUpdated dataset dimensions:")
print(df_weather.shape)


,time,temperature_2m,precipitation,is_rainy,temperature_category
0,2023-01-01 00:00:00+00:00,8.1,0.0,False,0-10°C
1,2023-01-01 01:00:00+00:00,8.2,0.0,False,0-10°C
2,2023-01-01 02:00:00+00:00,8.4,0.0,False,0-10°C
3,2023-01-01 03:00:00+00:00,8.0,0.0,False,0-10°C
4,2023-01-01 04:00:00+00:00,8.0,0.0,False,0-10°C
5,2023-01-01 05:00:00+00:00,8.1,0.0,False,0-10°C
6,2023-01-01 06:00:00+00:00,7.9,0.0,False,0-10°C
7,2023-01-01 07:00:00+00:00,7.3,0.0,False,0-10°C
8,2023-01-01 08:00:00+00:00,8.0,0.0,False,0-10°C
9,2023-01-01 09:00:00+00:00,10.2,0.0,False,10-20°C



Missing values in new variables:
is_rainy                0
temperature_category    0
dtype: int64

Updated dataset dimensions:
(26304, 8)



## 5. Saving the Cleaned Dataset

After completing the data cleaning and
feature engineering steps, the processed
weather dataset is saved as a CSV file.

The dataset contains:
- 26,304 hourly observations from 2023 to 2025
- Standardized UTC timestamps
- Five original weather variables
- Two additional weather variables
- No missing values
- No duplicate timestamps

The cleaned dataset will be used for integration
with the bicycle traffic data.

The original raw dataset remains unchanged.


In [9]:

# Define the output directory for processed data
output_dir = project_root / "data" / "processed"

# Create the directory if it does not exist
output_dir.mkdir(parents=True, exist_ok=True)

# Define the output file path
output_file = (
    output_dir /
    "weather_zurich_hourly_2023_2025_cleaned.csv"
)

# Validate the dataset before saving
assert len(df_weather) == 26304
assert df_weather["time"].is_unique
assert not df_weather.isnull().any().any()

# Save the cleaned dataset as a CSV file
df_weather.to_csv(
    output_file,
    index=False
)

# Display confirmation
print("Cleaned weather dataset saved successfully!")
print("File path:", output_file)
print("Number of rows:", len(df_weather))
print("Number of columns:", len(df_weather.columns))


Cleaned weather dataset saved successfully!
File path: C:\Sven\Studium\Fächer\2. Semester Hauptstudium\Data Analytics\Projekt\velo-wetter-zuerich-HS26\velo-wetter-zuerich-HS26\data\processed\weather_zurich_hourly_2023_2025_cleaned.csv
Number of rows: 26304
Number of columns: 8


In [10]:

# Reload the cleaned weather dataset
df_check = pd.read_csv(
    output_file,
    parse_dates=["time"]
)

# Ensure timestamps are in UTC
df_check["time"] = pd.to_datetime(
    df_check["time"],
    utc=True
)

# Display dataset information
print("Dataset dimensions:")
print(df_check.shape)

print("\nMissing values:")
print(df_check.isnull().sum())

print("\nDuplicate timestamps:")
print(df_check["time"].duplicated().sum())

print("\nFirst five rows:")
display(df_check.head())


Dataset dimensions:
(26304, 8)

Missing values:
time                    0
temperature_2m          0
precipitation           0
relative_humidity_2m    0
wind_speed_10m          0
cloud_cover             0
is_rainy                0
temperature_category    0
dtype: int64

Duplicate timestamps:
0

First five rows:


,time,temperature_2m,precipitation,relative_humidity_2m,wind_speed_10m,cloud_cover,is_rainy,temperature_category
0,2023-01-01 00:00:00+00:00,8.1,0.0,84,5.1,74,False,0-10°C
1,2023-01-01 01:00:00+00:00,8.2,0.0,83,4.6,48,False,0-10°C
2,2023-01-01 02:00:00+00:00,8.4,0.0,85,3.6,48,False,0-10°C
3,2023-01-01 03:00:00+00:00,8.0,0.0,86,2.9,61,False,0-10°C
4,2023-01-01 04:00:00+00:00,8.0,0.0,86,3.2,41,False,0-10°C
